In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/vivaansah/brain-tumor/data/Training/pituitary/Tr-pi_124.jpg
/kaggle/input/datasets/vivaansah/brain-tumor/data/Training/pituitary/Tr-pi_949.jpg
/kaggle/input/datasets/vivaansah/brain-tumor/data/Training/pituitary/Tr-pi_786.jpg
/kaggle/input/datasets/vivaansah/brain-tumor/data/Training/pituitary/Tr-pi_371.jpg
/kaggle/input/datasets/vivaansah/brain-tumor/data/Training/pituitary/Tr-pi_599.jpg
/kaggle/input/datasets/vivaansah/brain-tumor/data/Training/pituitary/Tr-pi_802.jpg
/kaggle/input/datasets/vivaansah/brain-tumor/data/Training/pituitary/Tr-pi_1323.jpg
/kaggle/input/datasets/vivaansah/brain-tumor/data/Training/pituitary/Tr-pi_1347.jpg
/kaggle/input/datasets/vivaansah/brain-tumor/data/Training/pituitary/Tr-pi_955.jpg
/kaggle/input/datasets/vivaansah/brain-tumor/data/Training/pituitary/Tr-pi_778.jpg
/kaggle/input/datasets/vivaansah/brain-tumor/data/Training/pituitary/Tr-pi_1354.jpg
/kaggle/input/datasets/vivaansah/brain-tumor/data/Training/pituitary/Tr-pi_470.jpg
/

In [2]:
!find /kaggle/input -maxdepth 3 -type d

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/vivaansah
/kaggle/input/datasets/vivaansah/brain-tumor


In [3]:
!mkdir -p /kaggle/working/src

In [4]:
!pip install -q timm tabulate

In [5]:
!find /kaggle/input/datasets/vivaansah/brain-tumor -maxdepth 3 -type d

/kaggle/input/datasets/vivaansah/brain-tumor
/kaggle/input/datasets/vivaansah/brain-tumor/data
/kaggle/input/datasets/vivaansah/brain-tumor/data/Training
/kaggle/input/datasets/vivaansah/brain-tumor/data/Training/pituitary
/kaggle/input/datasets/vivaansah/brain-tumor/data/Training/notumor
/kaggle/input/datasets/vivaansah/brain-tumor/data/Training/meningioma
/kaggle/input/datasets/vivaansah/brain-tumor/data/Training/glioma
/kaggle/input/datasets/vivaansah/brain-tumor/data/Testing
/kaggle/input/datasets/vivaansah/brain-tumor/data/Testing/pituitary
/kaggle/input/datasets/vivaansah/brain-tumor/data/Testing/notumor
/kaggle/input/datasets/vivaansah/brain-tumor/data/Testing/meningioma
/kaggle/input/datasets/vivaansah/brain-tumor/data/Testing/glioma


In [6]:
import os

DATA_DIR = "/kaggle/input/datasets/vivaansah/brain-tumor"

print("Dataset exists:", os.path.exists(DATA_DIR))
print("\nContents:")

for root, dirs, files in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:5]:
        print(f"{indent}  {file}")

Dataset exists: True

Contents:
brain-tumor/
  data/
    Training/
      pituitary/
        Tr-pi_124.jpg
        Tr-pi_949.jpg
        Tr-pi_786.jpg
        Tr-pi_371.jpg
        Tr-pi_599.jpg
      notumor/
        Tr-no_323.jpg
        Tr-no_86.jpg
        Tr-no_737.jpg
        Tr-no_452.jpg
        Tr-no_557.jpg
      meningioma/
        Tr-me_166.jpg
        Tr-me_1256.jpg
        Tr-me_115.jpg
        Tr-me_431.jpg
        Tr-me_971.jpg
      glioma/
        Tr-gl_1362.jpg
        Tr-gl_821.jpg
        Tr-gl_825.jpg
        Tr-gl_315.jpg
        Tr-gl_52.jpg
    Testing/
      pituitary/
        Te-pi_307.jpg
        Te-pi_269.jpg
        Te-pi_63.jpg
        Te-pi_259.jpg
        Te-pi_370.jpg
      notumor/
        Te-no_39.jpg
        Te-no_46.jpg
        Te-no_340.jpg
        Te-no_124.jpg
        Te-no_189.jpg
      meningioma/
        Te-me_251.jpg
        Te-me_175.jpg
        Te-me_60.jpg
        Te-aug-me_67.jpg
        Te-me_126.jpg
      glioma/
        Te-gl_16.jpg
  

In [7]:
!pip install -q timm torchmetrics scikit-learn seaborn

In [8]:
import torch
import timm
import sklearn

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [9]:
%%writefile /kaggle/working/main.py
import os
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from src.config import Config
from src.utils import set_seed, setup_logger
from src.dataset import get_dataloaders, BrainTumorDataset
from src.model import CNNBaseline, ViTBaseline, BasicHybridModel, ImprovedHybridModel
from src.train import train_model
from src.evaluate import evaluate_model, plot_training_history, calculate_multiclass_metrics, save_confusion_matrix
from src.explainability import visualize_cnn_gradcam, visualize_hybrid_attention
from torch.utils.data import Subset, DataLoader
from sklearn.svm import SVC

logger = setup_logger()


def load_checkpoint(model, checkpoint_path):
    """Loads a checkpoint safely onto the correct device (fixes CPU/GPU mismatch on Kaggle).
    Import this same helper from src.utils if you want train.py to share it too."""
    state_dict = torch.load(checkpoint_path, map_location=Config.DEVICE, weights_only=True)
    model.load_state_dict(state_dict)
    model = model.to(Config.DEVICE)
    return model


def parse_args():
    parser = argparse.ArgumentParser(description="Hybrid CNN-ViT Brain Tumor Classification Pipeline")
    parser.add_argument(
        '--mode', 
        type=str, 
        default='dry_run',
        choices=['train_eval', 'ablation', 'explain', 'dry_run'],
        help="Pipeline execution mode. 'dry_run' verifies pipeline quickly with tiny subsets."
    )
    parser.add_argument(
        '--model', 
        type=str, 
        default='improved_hybrid',
        choices=['cnn', 'vit', 'basic_hybrid', 'improved_hybrid', 'cnn_svm'],
        help="Model architecture to train/evaluate (used in 'train_eval' and 'explain' modes)."
    )
    parser.add_argument('--epochs', type=int, default=None, help="Override epochs.")
    parser.add_argument('--batch_size', type=int, default=None, help="Override batch size.")
    parser.add_argument('--lr', type=float, default=None, help="Override learning rate.")
    # parse_known_args so this survives Kaggle/Jupyter's injected -f kernel arg
    args, _ = parser.parse_known_args()
    return args


def extract_features(cnn_model, loader):
    """Extracts global average pooled feature vectors from a CNN baseline model."""
    cnn_model.eval()
    device = Config.DEVICE
    feats = []
    targets = []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            # Use forward_features from timm
            features = cnn_model.model.forward_features(imgs)
            # Apply adaptive pooling if feature map is 4D (B, C, H, W)
            if len(features.shape) == 4:
                features = F.adaptive_avg_pool2d(features, 1).flatten(1)
            elif len(features.shape) == 3: # For ViT 3D (B, N, C)
                features = features[:, 0]
                
            feats.append(features.cpu().numpy())
            targets.extend(labels.numpy())
            
    return np.concatenate(feats, axis=0), np.array(targets)


def train_eval_svm(cnn_model, train_loader, test_loader):
    """Trains a Support Vector Machine on CNN extracted features."""
    logger.info("Extracting training features using CNN baseline...")
    X_train, y_train = extract_features(cnn_model, train_loader)
    
    logger.info("Extracting testing features using CNN baseline...")
    X_test, y_test = extract_features(cnn_model, test_loader)
    
    logger.info(f"Training SVM on features (Train size: {X_train.shape[0]}, Feature dim: {X_train.shape[1]})...")
    svm_clf = SVC(kernel='rbf', C=1.0, probability=True, random_state=Config.SEED)
    svm_clf.fit(X_train, y_train)
    
    logger.info("Evaluating SVM on test set...")
    preds = svm_clf.predict(X_test)
    probs = svm_clf.predict_proba(X_test)
    
    metrics = calculate_multiclass_metrics(y_test, preds, probs)
    save_confusion_matrix(metrics['confusion_matrix'], "cnn_svm")
    
    # Save the SVM metrics to a report format
    logger.info(f"SVM Test Accuracy: {metrics['accuracy']:.4f}")
    return metrics


def get_class_weights(train_loader):
    """Calculates inverse frequency class weights to address minor imbalances after de-duplication."""
    labels = []
    for _, batch_labels in train_loader:
        labels.extend(batch_labels.numpy())
        
    counts = np.bincount(labels)
    total = sum(counts)
    # Inverse class frequencies
    weights = [total / (len(counts) * counts[i]) if counts[i] > 0 else 1.0 for i in range(len(counts))]
    weights = torch.tensor(weights, dtype=torch.float).to(Config.DEVICE)
    logger.info(f"Calculated class weights for loss function: {weights.cpu().numpy()}")
    return weights


def get_model_by_name(name):
    """Factory to instantiate models based on name string."""
    if name == 'cnn':
        return CNNBaseline()
    elif name == 'vit':
        return ViTBaseline()
    elif name == 'basic_hybrid':
        return BasicHybridModel()
    elif name == 'improved_hybrid':
        return ImprovedHybridModel()
    else:
        raise ValueError(f"Unknown model name: {name}")


def main():
    args = parse_args()
    set_seed(Config.SEED)
    Config.create_dirs()
    
    # Override configs if CLI arguments are provided (fixed: use "is not None" so --epochs 0 etc. work)
    if args.epochs is not None: Config.EPOCHS = args.epochs
    if args.batch_size is not None: Config.BATCH_SIZE = args.batch_size
    if args.lr is not None: Config.LR = args.lr
    
    # 1. Load Data
    train_loader, val_loader, test_loader = get_dataloaders()
    class_weights = get_class_weights(train_loader)
    
    # Handle dry_run mode to test the code correctness quickly
    if args.mode == 'dry_run':
        logger.info("\n" + "!"*40 + "\nDRY RUN MODE ENABLED - Training on tiny subset for 1 epoch\n" + "!"*40)
        # Create tiny subsets (e.g., 32 samples for training, 16 for val, 16 for test)
        train_sub = Subset(train_loader.dataset, range(min(32, len(train_loader.dataset))))
        val_sub = Subset(val_loader.dataset, range(min(16, len(val_loader.dataset))))
        test_sub = Subset(test_loader.dataset, range(min(16, len(test_loader.dataset))))
        
        train_loader = DataLoader(train_sub, batch_size=4, shuffle=True)
        val_loader = DataLoader(val_sub, batch_size=4, shuffle=False)
        test_loader = DataLoader(test_sub, batch_size=4, shuffle=False)
        
        Config.EPOCHS = 1  # 1 epoch is enough for verification
        
    # Execute according to mode
    if args.mode in ['train_eval', 'dry_run']:
        if args.model == 'cnn_svm':
            # Need to train CNN baseline first, or check if it exists
            cnn_checkpoint = os.path.join(Config.MODELS_DIR, "cnn_best.pth")
            cnn = CNNBaseline()
            if not os.path.exists(cnn_checkpoint):
                logger.info("CNN baseline checkpoint not found. Training CNN baseline first...")
                cnn, _ = train_model(cnn, train_loader, val_loader, "cnn", epochs=Config.EPOCHS, class_weights=class_weights)
            else:
                cnn = load_checkpoint(cnn, cnn_checkpoint)
                logger.info(f"Loaded trained CNN baseline from {cnn_checkpoint}")
                
            metrics = train_eval_svm(cnn, train_loader, test_loader)
            
            # Print table
            logger.info("\n" + "="*40 + "\nCNN-SVM Baseline Evaluation Complete\n" + "="*40)
        else:
            model = get_model_by_name(args.model)
            model, history = train_model(
                model, 
                train_loader, 
                val_loader, 
                args.model, 
                epochs=Config.EPOCHS, 
                lr=Config.LR, 
                class_weights=class_weights
            )
            
            # Plot loss/acc curves
            if args.mode != 'dry_run':
                plot_training_history(history, args.model)
                
            metrics, _, _, _ = evaluate_model(model, test_loader, args.model)
            
    elif args.mode == 'ablation':
        logger.info("\n=== Starting Comparative Ablation Study ===")
        # Run CNN, ViT, Basic Hybrid, and Improved Hybrid, and CNN-SVM
        results = []
        
        models_to_test = ['cnn', 'vit', 'basic_hybrid', 'improved_hybrid']
        
        for m_name in models_to_test:
            logger.info(f"\n--- Training/Evaluating Model: {m_name} ---")
            model = get_model_by_name(m_name)
            model, history = train_model(
                model, 
                train_loader, 
                val_loader, 
                m_name, 
                epochs=Config.EPOCHS, 
                lr=Config.LR, 
                class_weights=class_weights
            )
            plot_training_history(history, m_name)
            metrics, _, _, _ = evaluate_model(model, test_loader, m_name)
            
            results.append({
                'Model': m_name.replace('_', ' ').upper(),
                'Accuracy': metrics['accuracy'],
                'Precision': metrics['precision_weighted'],
                'Recall': metrics['recall_weighted'],
                'F1': metrics['f1_weighted'],
                'AUC': metrics['roc_auc_weighted']
            })
            
        # Add SVM baseline (uses the trained CNN baseline)
        logger.info("\n--- Training/Evaluating Model: CNN-SVM baseline ---")
        cnn_checkpoint = os.path.join(Config.MODELS_DIR, "cnn_best.pth")
        cnn = CNNBaseline()
        if os.path.exists(cnn_checkpoint):
            cnn = load_checkpoint(cnn, cnn_checkpoint)
            logger.info(f"Loaded trained CNN baseline from {cnn_checkpoint} for SVM feature extraction.")
        else:
            logger.info("CNN baseline checkpoint not found. Training CNN baseline first...")
            cnn, _ = train_model(cnn, train_loader, val_loader, "cnn", epochs=Config.EPOCHS, class_weights=class_weights)
            
        svm_metrics = train_eval_svm(cnn, train_loader, test_loader)
        
        results.append({
            'Model': 'CNN-SVM BASELINE',
            'Accuracy': svm_metrics['accuracy'],
            'Precision': svm_metrics['precision_weighted'],
            'Recall': svm_metrics['recall_weighted'],
            'F1': svm_metrics['f1_weighted'],
            'AUC': svm_metrics['roc_auc_weighted']
        })
        
        # Save results to a comparison report file (CSV)
        df_results = pd.DataFrame(results)
        report_path_csv = os.path.join(Config.REPORTS_DIR, "ablation_study_results.csv")
        df_results.to_csv(report_path_csv, index=False)
        
        # Safe markdown formatting
        try:
            markdown_table = df_results.to_markdown(index=False)
        except ImportError:
            markdown_table = df_results.to_string(index=False)
            
        # Save results to a comparison report file (Markdown)
        report_path_md = os.path.join(Config.REPORTS_DIR, "ablation_study_results.md")
        with open(report_path_md, 'w', encoding='utf-8') as f:
            f.write("# Ablation Study Comparative Results\n\n")
            f.write(f"This report compares the performance of CNN, ViT, Hybrid, and CNN-SVM models on the Brain Tumor classification task.\n\n")
            if hasattr(df_results, 'to_markdown'):
                f.write(df_results.to_markdown(index=False))
            else:
                f.write("```\n" + df_results.to_string(index=False) + "\n```")
        
        print("\n" + "="*70)
        print("ABLATION STUDY COMPARATIVE RESULTS")
        print("="*70)
        print(markdown_table)
        print("="*70)
        logger.info(f"Ablation study comparative report saved to {report_path_csv} and {report_path_md}")
        
    elif args.mode == 'explain':
        logger.info(f"Generating interpretability visualizations for: {args.model}")
        checkpoint_path = os.path.join(Config.MODELS_DIR, f"{args.model}_best.pth")
        if not os.path.exists(checkpoint_path):
            raise FileNotFoundError(f"Trained checkpoint not found: {checkpoint_path}. Please train the model first.")
            
        model = get_model_by_name(args.model)
        model = load_checkpoint(model, checkpoint_path)
        
        # We will pick a few representative test samples from each category (glioma, meningioma, pituitary, notumor)
        # to generate explainability plots.
        test_dir = os.path.join(Config.DATA_DIR, "Testing")
        
        for cls in sorted(Config.CLASSES):
            cls_dir = os.path.join(test_dir, cls)
            if not os.path.exists(cls_dir): continue
            
            # Pick first available image in that class
            files = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            if len(files) == 0: continue
            
            img_path = os.path.join(cls_dir, files[0])
            save_name = f"{args.model}_{cls}_explain.png"
            save_path = os.path.join(Config.PLOTS_DIR, save_name)
            
            logger.info(f"Processing explainability for Class '{cls}', file: {files[0]}")
            if args.model == 'cnn':
                visualize_cnn_gradcam(model, img_path, save_path)
            elif args.model == 'improved_hybrid':
                visualize_hybrid_attention(model, img_path, save_path)
            else:
                logger.info(f"Model '{args.model}' explainability visualization is not supported (CNN or Improved Hybrid only).")
                break
                
        logger.info("Explainability visualizations completed successfully.")

if __name__ == '__main__':
    main()

Writing /kaggle/working/main.py


In [10]:
%%writefile /kaggle/working/src/config.py
import os
import torch

class Config:
    # Random Seed
    SEED = 42

    # Paths -- Kaggle-adapted
    # /kaggle/input is READ-ONLY: dataset lives here.
    # /kaggle/working is the only writable, persistent-per-session location: all outputs go here.
    #DATA_DIR = "/kaggle/input/datasets/vivaansah/brain-tumor"  # <-- CONFIRM this with `!find /kaggle/input -maxdepth 3 -type d` first
    DATA_DIR = "/kaggle/input/datasets/vivaansah/brain-tumor/data"
    OUTPUT_DIR = "/kaggle/working/outputs"

    MODELS_DIR = os.path.join(OUTPUT_DIR, "models")
    PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")
    CM_DIR = os.path.join(OUTPUT_DIR, "confusion_matrices")
    REPORTS_DIR = os.path.join(OUTPUT_DIR, "reports")

    # Dataset Settings
    CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
    NUM_CLASSES = len(CLASSES)
    VAL_SPLIT = 0.2
    DEDUPLICATE = True
    # Was a CPU-speed limit (150/50 per class) for local testing on a laptop.
    # On Kaggle's GPU you almost certainly want the FULL dataset -> set to None.
    # Drop these back to small ints only if you want a fast dry-run without using --mode dry_run.
    MAX_SAMPLES_PER_CLASS_TRAIN = None
    MAX_SAMPLES_PER_CLASS_TEST = None

    # Preprocessing
    IMAGE_SIZE = (224, 224)
    # Means and Stds for standard ImageNet normalization
    NORM_MEAN = [0.485, 0.456, 0.406]
    NORM_STD = [0.229, 0.224, 0.225]

    # Model Parameters
    CNN_BACKBONE = 'resnet18'  # or 'efficientnet_b0'
    ViT_BACKBONE = 'vit_tiny_patch16_224' # or 'vit_small_patch16_224'
    
    # Custom Hybrid Model Configuration
    HYBRID_EMBED_DIM = 256
    HYBRID_NUM_HEADS = 8
    HYBRID_NUM_LAYERS = 4
    HYBRID_MLP_DIM = 512
    HYBRID_DROPOUT = 0.3

    # Training Hyperparameters
    BATCH_SIZE = 16
    LR = 1e-4
    WEIGHT_DECAY = 1e-2
    EPOCHS = 5
    EARLY_STOPPING_PATIENCE = 5
    FINE_TUNE_EPOCHS = 2 # Epochs to freeze CNN backbone during initial phase
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

    @classmethod
    def create_dirs(cls):
        """Creates the output directories if they do not exist."""
        for d in [cls.OUTPUT_DIR, cls.MODELS_DIR, cls.PLOTS_DIR, cls.CM_DIR, cls.REPORTS_DIR]:
            os.makedirs(d, exist_ok=True)

Writing /kaggle/working/src/config.py


In [11]:

%%writefile /kaggle/working/src/train.py
import os
import torch
import numpy as np
import torch.nn as nn
from torch.amp import autocast, GradScaler
from sklearn.metrics import f1_score
from src.config import Config
from src.utils import setup_logger

logger = setup_logger()

class EarlyStopping:
    """Early stops the training if validation loss doesn't improve after a given patience."""
    def __init__(self, patience=5, verbose=False, delta=0, path='checkpoint.pth'):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = float('inf')
        self.delta = delta
        self.path = path

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            logger.info(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        """Saves model when validation loss decreases."""
        if self.verbose:
            logger.info(f"Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}). Saving model...")
        os.makedirs(os.path.dirname(self.path), exist_ok=True)
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss


def train_model(model, train_loader, val_loader, model_name="hybrid_model", epochs=Config.EPOCHS, lr=Config.LR, class_weights=None):
    """Trains a given PyTorch model, logging training and validation metrics, and implementing early stopping."""
    device = Config.DEVICE
    model = model.to(device)
    
    # Loss Function (with optional class weights to handle imbalance)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    
    # Check if model supports gradual fine-tuning (has both cnn backbone and transformer blocks)
    is_hybrid = hasattr(model, 'cnn') and hasattr(model, 'transformer')
    
    # Freezing the CNN backbone initially for gradual fine-tuning
    if is_hybrid and Config.FINE_TUNE_EPOCHS > 0:
        logger.info(f"Gradual Fine-Tuning: Freezing CNN backbone for the first {Config.FINE_TUNE_EPOCHS} epochs.")
        for param in model.cnn.parameters():
            param.requires_grad = False
            
    # Optimizer and Scheduler
    if is_hybrid and Config.FINE_TUNE_EPOCHS > 0:
        # Initially only optimize non-backbone parameters
        trainable_params = [p for n, p in model.named_parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=Config.WEIGHT_DECAY)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=Config.WEIGHT_DECAY)
        
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    # Mixed precision scaler
    scaler = GradScaler('cuda', enabled=(device == 'cuda'))
    
    # Paths for checkpoints
    checkpoint_path = os.path.join(Config.MODELS_DIR, f"{model_name}_best.pth")
    early_stopping = EarlyStopping(patience=Config.EARLY_STOPPING_PATIENCE, verbose=True, path=checkpoint_path)
    
    # Tracking history
    history = {
        'train_loss': [], 'train_acc': [], 'train_f1': [],
        'val_loss': [], 'val_acc': [], 'val_f1': []
    }
    
    logger.info(f"Starting training for {model_name} on device: {device}")
    
    for epoch in range(epochs):
        # Unfreeze backbone after Config.FINE_TUNE_EPOCHS
        if is_hybrid and Config.FINE_TUNE_EPOCHS > 0 and epoch == Config.FINE_TUNE_EPOCHS:
            logger.info("Gradual Fine-Tuning: Unfreezing CNN backbone for full fine-tuning.")
            for param in model.cnn.parameters():
                param.requires_grad = True
            
            # Recreate optimizer to include CNN parameters with a lower learning rate
            optimizer = torch.optim.AdamW([
                {'params': model.cnn.parameters(), 'lr': lr * 0.1},  # lower learning rate for pretrained backbone
                {'params': [p for n, p in model.named_parameters() if 'cnn' not in n], 'lr': lr}
            ], weight_decay=Config.WEIGHT_DECAY)
            
            # Recreate scheduler for remaining epochs
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs - epoch)
        # --- TRAINING PHASE ---
        model.train()
        train_loss = 0.0
        train_preds = []
        train_targets = []
        
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            
            # Forward pass with mixed precision
            with autocast('cuda', enabled=(device == 'cuda')):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                
            # Backward pass & Optimizer Step
            if device == 'cuda':
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
                
            train_loss += loss.item() * imgs.size(0)
            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_targets.extend(labels.cpu().numpy())
            
        # Update LR scheduler
        scheduler.step()
        
        epoch_train_loss = train_loss / len(train_loader.dataset)
        epoch_train_acc = (sum(np.array(train_preds) == np.array(train_targets)) / len(train_targets))
        epoch_train_f1 = f1_score(train_targets, train_preds, average='weighted')
        
        # --- VALIDATION PHASE ---
        model.eval()
        val_loss = 0.0
        val_preds = []
        val_targets = []
        
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                
                with autocast('cuda', enabled=(device == 'cuda')):
                    outputs = model(imgs)
                    loss = criterion(outputs, labels)
                    
                val_loss += loss.item() * imgs.size(0)
                preds = torch.argmax(outputs, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(labels.cpu().numpy())
                
        epoch_val_loss = val_loss / len(val_loader.dataset)
        epoch_val_acc = (sum(np.array(val_preds) == np.array(val_targets)) / len(val_targets))
        epoch_val_f1 = f1_score(val_targets, val_preds, average='weighted')
        
        # Log Epoch Summary
        logger.info(
            f"Epoch {epoch+1:02d}/{epochs:02d} | "
            f"Train Loss: {epoch_train_loss:.4f} - Acc: {epoch_train_acc:.4f} - F1: {epoch_train_f1:.4f} | "
            f"Val Loss: {epoch_val_loss:.4f} - Acc: {epoch_val_acc:.4f} - F1: {epoch_val_f1:.4f}"
        )
        
        # Record metrics
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)
        history['train_f1'].append(epoch_train_f1)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)
        history['val_f1'].append(epoch_val_f1)
        
        # Check early stopping
        early_stopping(epoch_val_loss, model)
        if early_stopping.early_stop:
            logger.info("Early stopping triggered. Training stopped.")
            break
            
    # Load the best saved model state before returning
    # Fixed: map_location=device (checkpoint could've been saved on a different device
    # in a resumed session) + weights_only=True (avoids PyTorch 2.x FutureWarning/future break)
    if os.path.exists(checkpoint_path):
        model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
        model = model.to(device)
        logger.info(f"Loaded best validation model checkpoint from {checkpoint_path}")
        
    return model, history

Writing /kaggle/working/src/train.py


In [12]:
%%writefile /kaggle/working/src/explainability.py
import os
import cv2
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from src.config import Config
from src.utils import setup_logger

logger = setup_logger()

class GradCAM:
    """Helper class to compute Grad-CAM heatmap for a target CNN layer."""
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.handlers = []
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0]

        # Register forward and backward hooks on the target layer
        # Note: register_full_backward_hook (not the deprecated register_backward_hook)
        # is required on modern PyTorch versions like the one on Kaggle.
        self.handlers.append(self.target_layer.register_forward_hook(forward_hook))
        self.handlers.append(self.target_layer.register_full_backward_hook(backward_hook))

    def remove_hooks(self):
        for handler in self.handlers:
            handler.remove()

    def generate_heatmap(self, input_tensor, class_idx=None):
        self.model.eval()
        output = self.model(input_tensor)
        
        if class_idx is None:
            class_idx = torch.argmax(output, dim=1).item()
            
        self.model.zero_grad()
        loss = output[0, class_idx]
        loss.backward()
        
        # Pool the gradients across channels
        # gradients shape: (B, C, H, W) -> activations shape: (B, C, H, W)
        pooled_gradients = torch.mean(self.gradients, dim=[0, 2, 3])
        activations = self.activations[0]
        
        # Multiply each channel by its corresponding gradient weight
        for i in range(activations.shape[0]):
            activations[i] *= pooled_gradients[i]
            
        # Average across channels to get the heat map
        heatmap = torch.mean(activations, dim=0).detach().cpu().numpy()
        
        # Apply ReLU to keep positive influence only, and normalize
        heatmap = np.maximum(heatmap, 0)
        if heatmap.max() > 0:
            heatmap /= heatmap.max()
            
        return heatmap, class_idx


def overlay_heatmap(img_path, heatmap, alpha=0.4, colormap=cv2.COLORMAP_JET):
    """Overlays a 2D heatmap on top of the original image."""
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Resize heatmap to match original image size
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    
    # Convert heatmap to RGB colors
    heatmap_color = np.uint8(255 * heatmap_resized)
    heatmap_color = cv2.applyColorMap(heatmap_color, colormap)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    
    # Combine original image and colored heatmap
    overlayed = cv2.addWeighted(img, 1.0 - alpha, heatmap_color, alpha, 0)
    return overlayed, img


def save_explainability_plot(original_img, map_overlay, title, save_path, disclaimer=True):
    """Saves explainability plots showing the original image and overlay side-by-side."""
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    
    # Original Image
    axes[0].imshow(original_img)
    axes[0].set_title("Original Image")
    axes[0].axis('off')
    
    # Explainability Overlay
    axes[1].imshow(map_overlay)
    axes[1].set_title(title)
    axes[1].axis('off')
    
    if disclaimer:
        fig.text(
            0.5, 0.02, 
            "DISCLAIMER: This visualization is for model interpretability and research purposes, NOT for clinical diagnosis.", 
            ha='center', fontsize=9, color='red', weight='bold', style='italic'
        )
        
    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    logger.info(f"Saved explainability visualization to {save_path}")


def visualize_cnn_gradcam(model, image_path, save_path, class_idx=None):
    """Computes and plots Grad-CAM for the CNN baseline model."""
    from src.preprocessing import MedicalPreprocessing
    
    # Preprocess image
    img_tensor = MedicalPreprocessing.preprocess_image(image_path, is_training=False).unsqueeze(0).to(Config.DEVICE)
    
    # Find the target layer for ResNet-18 (layer4) or EfficientNet-B0 (act2 of last block)
    target_layer = None
    if hasattr(model.model, 'layer4'):
        # For ResNet models
        target_layer = model.model.layer4[-1]
    elif hasattr(model.model, 'conv_head'):
        # For EfficientNet models
        target_layer = model.model.conv_head
    else:
        logger.error("Could not determine final conv layer for Grad-CAM.")
        return
        
    gcam = GradCAM(model, target_layer)
    try:
        heatmap, pred_idx = gcam.generate_heatmap(img_tensor, class_idx)
        overlay, img = overlay_heatmap(image_path, heatmap)
        pred_class = sorted(Config.CLASSES)[pred_idx]
        title = f"Grad-CAM (Predicted: {pred_class})"
        save_explainability_plot(img, overlay, title, save_path)
    finally:
        gcam.remove_hooks()


def visualize_hybrid_attention(model, image_path, save_path, class_idx=None):
    """Visualizes attention maps of the Improved Hybrid model side by side."""
    from src.preprocessing import MedicalPreprocessing
    
    # Preprocess image
    img_tensor = MedicalPreprocessing.preprocess_image(image_path, is_training=False).unsqueeze(0).to(Config.DEVICE)
    
    # Forward pass storing attention weights
    model.eval()
    with torch.no_grad():
        output = model(img_tensor, store_attention=True)
        pred_idx = torch.argmax(output, dim=1).item()
        
    pred_class = sorted(Config.CLASSES)[pred_idx]
    
    # Extract attention weights from final Transformer layer
    # attn_weights shape: (1, num_heads, seq_len, seq_len)
    attn_weights = model.transformer[-1].attn.attention_weights
    
    if attn_weights is None:
        logger.error("No attention weights found in model. Ensure store_attention=True in forward pass.")
        return
        
    # Average across attention heads (dim=1)
    # Get CLS token attention (index 0) to all patch tokens (indices 1 to 245)
    cls_attn = attn_weights[0, :, 0, 1:].mean(dim=0).numpy()  # shape: (245,)
    
    # Split back into Stage 3 (196 tokens -> 14x14) and Stage 4 (49 tokens -> 7x7)
    attn_s3 = cls_attn[:196].reshape(14, 14)
    attn_s4 = cls_attn[196:].reshape(7, 7)
    
    # Normalize each attention map to [0, 1]
    if attn_s3.max() > 0:
        attn_s3 /= attn_s3.max()
    if attn_s4.max() > 0:
        attn_s4 /= attn_s4.max()
        
    # Read original image
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Resize and color maps
    h_orig, w_orig, _ = img.shape
    s3_resized = cv2.resize(attn_s3, (w_orig, h_orig))
    s4_resized = cv2.resize(attn_s4, (w_orig, h_orig))
    
    # Apply colormaps
    s3_color = cv2.applyColorMap(np.uint8(255 * s3_resized), cv2.COLORMAP_JET)
    s3_color = cv2.cvtColor(s3_color, cv2.COLOR_BGR2RGB)
    
    s4_color = cv2.applyColorMap(np.uint8(255 * s4_resized), cv2.COLORMAP_JET)
    s4_color = cv2.cvtColor(s4_color, cv2.COLOR_BGR2RGB)
    
    # Overlay on original image
    alpha = 0.4
    overlay_s3 = cv2.addWeighted(img, 1.0 - alpha, s3_color, alpha, 0)
    overlay_s4 = cv2.addWeighted(img, 1.0 - alpha, s4_color, alpha, 0)
    
    # Plot side-by-side: Original, Stage 3 Attention (14x14), Stage 4 Attention (7x7)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(img)
    axes[0].set_title("Original Image")
    axes[0].axis('off')
    
    axes[1].imshow(overlay_s3)
    axes[1].set_title(f"S3 Attn Map (14x14) - Pred: {pred_class}")
    axes[1].axis('off')
    
    axes[2].imshow(overlay_s4)
    axes[2].set_title(f"S4 Attn Map (7x7) - Pred: {pred_class}")
    axes[2].axis('off')
    
    fig.text(
        0.5, 0.02, 
        "DISCLAIMER: This visualization is for model interpretability and research purposes, NOT for clinical diagnosis.", 
        ha='center', fontsize=10, color='red', weight='bold', style='italic'
    )
    
    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    logger.info(f"Saved Hybrid Transformer attention maps to {save_path}")

Writing /kaggle/working/src/explainability.py


In [13]:
%%writefile /kaggle/working/src/dataset.py
import os
import hashlib
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from collections import Counter
from src.config import Config
from src.preprocessing import MedicalPreprocessing

class BrainTumorDataset(Dataset):
    """Custom PyTorch Dataset for loading brain tumor MRI scans with on-the-fly preprocessing."""
    def __init__(self, file_paths, labels, is_training=False):
        self.file_paths = file_paths
        self.labels = labels
        self.is_training = is_training

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        label = self.labels[idx]
        
        try:
            img_tensor = MedicalPreprocessing.preprocess_image(path, is_training=self.is_training)
        except Exception as e:
            # Fallback to dummy tensor if there is a reading issue
            print(f"Error loading image {path}: {str(e)}")
            img_tensor = torch.zeros((3, Config.IMAGE_SIZE[0], Config.IMAGE_SIZE[1]))
            
        label_tensor = torch.tensor(label, dtype=torch.long)
        return img_tensor, label_tensor

def compute_md5(file_path):
    """Computes MD5 hash of a file to check for exact duplicates."""
    hasher = hashlib.md5()
    with open(file_path, 'rb') as f:
        buf = f.read()
        hasher.update(buf)
    return hasher.hexdigest()

def get_unique_files(root_dir, deduplicate=True, max_samples_per_class=None):
    """Scans root_dir for image files, identifies classes, and optionally filters duplicates using MD5 hashes.
    
    Returns:
        list of file paths, list of labels (integers), and list of duplicate paths removed.
    """
    file_paths = []
    labels = []
    seen_hashes = set()
    duplicates_removed = []
    
    # Sort classes to guarantee consistent label ordering
    classes = sorted(Config.CLASSES)
    class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
    
    for cls in classes:
        cls_dir = os.path.join(root_dir, cls)
        if not os.path.exists(cls_dir):
            continue
        
        class_count = 0
        for filename in sorted(os.listdir(cls_dir)):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
                full_path = os.path.join(cls_dir, filename)
                label_idx = class_to_idx[cls]
                
                if deduplicate:
                    md5 = compute_md5(full_path)
                    if md5 in seen_hashes:
                        duplicates_removed.append(full_path)
                        continue
                    seen_hashes.add(md5)
                
                file_paths.append(full_path)
                labels.append(label_idx)
                
                class_count += 1
                if max_samples_per_class is not None and class_count >= max_samples_per_class:
                    break
                
    return file_paths, labels, duplicates_removed

def get_dataloaders():
    """Generates train, validation, and test PyTorch DataLoaders.
    Handles de-duplication, logging of dataset statistics, and stratified split of training.
    """
    train_dir = os.path.join(Config.DATA_DIR, "Training")
    test_dir = os.path.join(Config.DATA_DIR, "Testing")
    
    # Load and de-duplicate files
    train_paths, train_labels, train_dups = get_unique_files(
        train_dir, 
        deduplicate=Config.DEDUPLICATE, 
        max_samples_per_class=Config.MAX_SAMPLES_PER_CLASS_TRAIN
    )
    test_paths, test_labels, test_dups = get_unique_files(
        test_dir, 
        deduplicate=Config.DEDUPLICATE, 
        max_samples_per_class=Config.MAX_SAMPLES_PER_CLASS_TEST
    )
    
    print("\n" + "="*50)
    print("DATASET ANALYSIS & DE-DUPLICATION SUMMARY")
    print("="*50)
    print(f"Training images found (before de-dup): {len(train_paths) + len(train_dups)}")
    print(f"Training images unique: {len(train_paths)}")
    print(f"Training duplicates removed: {len(train_dups)}")
    
    print(f"Testing images found (before de-dup): {len(test_paths) + len(test_dups)}")
    print(f"Testing images unique: {len(test_paths)}")
    print(f"Testing duplicates removed: {len(test_dups)}")
    
    # Report class distributions on unique files
    train_dist = Counter(train_labels)
    test_dist = Counter(test_labels)
    
    print("\nClass Distribution (De-duplicated):")
    print(f"{'Class Name':<15} | {'Training Count':<15} | {'Testing Count':<15}")
    print("-" * 55)
    for idx, cls in enumerate(sorted(Config.CLASSES)):
        print(f"{cls:<15} | {train_dist.get(idx, 0):<15} | {test_dist.get(idx, 0):<15}")
    print("="*50 + "\n")
    
    # Perform Stratified Train-Validation Split (80% Train, 20% Val)
    # Splitting unique training paths to avoid data leakage
    tr_paths, val_paths, tr_labels, val_labels = train_test_split(
        train_paths, 
        train_labels, 
        test_size=Config.VAL_SPLIT, 
        random_state=Config.SEED,
        stratify=train_labels
    )
    
    print(f"Stratified Split:")
    print(f"  Training Set Size:   {len(tr_paths)}")
    print(f"  Validation Set Size: {len(val_paths)}")
    print(f"  Testing Set Size:    {len(test_paths)}")
    
    # Create PyTorch datasets
    train_dataset = BrainTumorDataset(tr_paths, tr_labels, is_training=True)
    val_dataset = BrainTumorDataset(val_paths, val_labels, is_training=False)
    test_dataset = BrainTumorDataset(test_paths, test_labels, is_training=False)
    
    # Create loaders
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, pin_memory=True)
    
    return train_loader, val_loader, test_loader


Writing /kaggle/working/src/dataset.py


In [14]:
%%writefile /kaggle/working/src/evaluate.py
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, precision_recall_curve, auc
)
from src.config import Config
from src.utils import setup_logger

logger = setup_logger()

def calculate_multiclass_metrics(y_true, y_pred, y_prob):
    """Calculates comprehensive multiclass classification metrics including per-class specificity."""
    # Standard metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision_weighted = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall_weighted = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Calculate specificity per class
    # Specificity = TN / (TN + FP)
    num_classes = cm.shape[0]
    specificities = []
    class_weights = []
    
    for i in range(num_classes):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - (tp + fp + fn)
        
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specificities.append(specificity)
        class_weights.append(cm[i, :].sum())  # Support for class i
        
    specificity_macro = np.mean(specificities)
    specificity_weighted = np.average(specificities, weights=class_weights)

    # ROC AUC (One-vs-Rest)
    try:
        roc_auc_macro = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
        roc_auc_weighted = roc_auc_score(y_true, y_prob, multi_class='ovr', average='weighted')
    except Exception as e:
        logger.warning(f"Could not compute ROC-AUC: {str(e)}")
        roc_auc_macro, roc_auc_weighted = 0.0, 0.0

    # PR AUC (One-vs-Rest)
    pr_aucs = []
    for i in range(num_classes):
        y_true_binary = (np.array(y_true) == i).astype(int)
        y_prob_class = y_prob[:, i]
        precision_curve, recall_curve, _ = precision_recall_curve(y_true_binary, y_prob_class)
        pr_aucs.append(auc(recall_curve, precision_curve))
        
    pr_auc_macro = np.mean(pr_aucs)
    pr_auc_weighted = np.average(pr_aucs, weights=class_weights)

    # Compile all metrics into a dictionary
    metrics = {
        'accuracy': accuracy,
        'precision_macro': precision_macro,
        'precision_weighted': precision_weighted,
        'recall_macro': recall_macro,
        'recall_weighted': recall_weighted,
        'specificity_macro': specificity_macro,
        'specificity_weighted': specificity_weighted,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'roc_auc_macro': roc_auc_macro,
        'roc_auc_weighted': roc_auc_weighted,
        'pr_auc_macro': pr_auc_macro,
        'pr_auc_weighted': pr_auc_weighted,
        'confusion_matrix': cm,
        'per_class_specificity': specificities,
        'per_class_f1': f1_score(y_true, y_pred, average=None, zero_division=0),
        'per_class_precision': precision_score(y_true, y_pred, average=None, zero_division=0),
        'per_class_recall': recall_score(y_true, y_pred, average=None, zero_division=0)
    }
    
    return metrics


def evaluate_model(model, loader, model_name="hybrid_model"):
    """Runs prediction on loader and computes detailed classification metrics."""
    device = Config.DEVICE
    model = model.to(device)
    model.eval()
    
    y_true = []
    y_pred = []
    y_prob = []
    
    logger.info(f"Evaluating {model_name} on test set...")
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            
            y_true.extend(labels.numpy())
            y_pred.extend(preds)
            y_prob.extend(probs)
            
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)
    
    metrics = calculate_multiclass_metrics(y_true, y_pred, y_prob)
    
    # Log global metrics
    logger.info(f"\n=================== {model_name.upper()} RESULTS ===================")
    logger.info(f"Accuracy:            {metrics['accuracy']:.4f}")
    logger.info(f"Precision (weighted): {metrics['precision_weighted']:.4f} (macro: {metrics['precision_macro']:.4f})")
    logger.info(f"Recall (Sensitivity): {metrics['recall_weighted']:.4f} (macro: {metrics['recall_macro']:.4f})")
    logger.info(f"Specificity:         {metrics['specificity_weighted']:.4f} (macro: {metrics['specificity_macro']:.4f})")
    logger.info(f"F1 Score (weighted):  {metrics['f1_weighted']:.4f} (macro: {metrics['f1_macro']:.4f})")
    logger.info(f"ROC-AUC (weighted):   {metrics['roc_auc_weighted']:.4f} (macro: {metrics['roc_auc_macro']:.4f})")
    logger.info(f"PR-AUC (weighted):    {metrics['pr_auc_weighted']:.4f} (macro: {metrics['pr_auc_macro']:.4f})")
    logger.info("=====================================================")
    
    # Save results
    save_confusion_matrix(metrics['confusion_matrix'], model_name)
    
    # Run failure analysis
    try:
        generate_failure_analysis(y_true, y_pred, y_prob, loader, model_name)
    except Exception as e:
        logger.warning(f"Could not generate failure analysis: {str(e)}")
        
    return metrics, y_true, y_pred, y_prob


def save_confusion_matrix(cm, model_name):
    """Plots and saves the confusion matrix to the outputs folder."""
    plt.figure(figsize=(8, 6))
    classes = sorted(Config.CLASSES)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes, cbar=False)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    
    path = os.path.join(Config.CM_DIR, f"{model_name}_cm.png")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    plt.savefig(path, dpi=300)
    plt.close()
    logger.info(f"Saved confusion matrix plot to {path}")


def plot_training_history(history, model_name):
    """Plots and saves loss and accuracy history curves."""
    epochs = len(history['train_loss'])
    x = range(1, epochs + 1)
    
    plt.figure(figsize=(12, 5))
    
    # Loss Curve
    plt.subplot(1, 2, 1)
    plt.plot(x, history['train_loss'], label='Train Loss', color='blue', marker='o')
    plt.plot(x, history['val_loss'], label='Val Loss', color='orange', marker='s')
    plt.title('Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    # Accuracy Curve
    plt.subplot(1, 2, 2)
    plt.plot(x, history['train_acc'], label='Train Acc', color='blue', marker='o')
    plt.plot(x, history['val_acc'], label='Val Acc', color='orange', marker='s')
    plt.title('Accuracy Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    path = os.path.join(Config.PLOTS_DIR, f"{model_name}_history.png")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    plt.savefig(path, dpi=300)
    plt.close()
    logger.info(f"Saved training history curves to {path}")


def get_loader_filepaths(loader):
    """Safely extracts image file paths from DataLoader datasets, including Subsets."""
    dataset = loader.dataset
    if hasattr(dataset, 'file_paths'):
        return dataset.file_paths
    elif hasattr(dataset, 'dataset') and hasattr(dataset, 'indices'):
        return [dataset.dataset.file_paths[idx] for idx in dataset.indices]
    return None


def generate_failure_analysis(y_true, y_pred, y_prob, loader, model_name):
    """Performs deep failure analysis on misclassified examples, confidence distributions, and saves a report."""
    filepaths = get_loader_filepaths(loader)
    classes = sorted(Config.CLASSES)
    
    # 1. Identify correct vs incorrect
    correct_mask = (y_true == y_pred)
    incorrect_mask = ~correct_mask
    
    # Confidences for predicted classes
    confidences = np.array([probs[pred] for probs, pred in zip(y_prob, y_pred)])
    
    avg_conf_correct = np.mean(confidences[correct_mask]) if correct_mask.sum() > 0 else 0.0
    avg_conf_incorrect = np.mean(confidences[incorrect_mask]) if incorrect_mask.sum() > 0 else 0.0
    
    # 2. Extract misclassified list
    failures = []
    if filepaths is not None:
        for idx in np.where(incorrect_mask)[0]:
            failures.append({
                'filepath': filepaths[idx],
                'filename': os.path.basename(filepaths[idx]),
                'true_class': classes[y_true[idx]],
                'pred_class': classes[y_pred[idx]],
                'confidence': confidences[idx],
                'probabilities': {classes[i]: float(y_prob[idx, i]) for i in range(len(classes))}
            })
            
    # Sort failures by confidence (descending) to find the most "confident-but-wrong" cases
    failures = sorted(failures, key=lambda x: x['confidence'], reverse=True)
    
    # 3. False Positives / False Negatives per class
    fp_counts = {cls: 0 for cls in classes}
    fn_counts = {cls: 0 for cls in classes}
    
    for idx in range(len(y_true)):
        t_cls = classes[y_true[idx]]
        p_cls = classes[y_pred[idx]]
        if t_cls != p_cls:
            fp_counts[p_cls] += 1
            fn_counts[t_cls] += 1
            
    # 4. Generate report content
    report = []
    report.append("="*70)
    report.append(f"FAILURE ANALYSIS AND ROBUSTNESS REPORT: {model_name.upper()}")
    report.append("="*70)
    report.append(f"Total Test Samples: {len(y_true)}")
    report.append(f"Correct Predictions: {correct_mask.sum()}")
    report.append(f"Incorrect Predictions: {incorrect_mask.sum()}")
    if len(y_true) > 0:
        report.append(f"Error Rate: {incorrect_mask.sum() / len(y_true) * 100:.2f}%")
    else:
        report.append("Error Rate: N/A")
    report.append(f"Average Confidence (Correct): {avg_conf_correct:.4f}")
    report.append(f"Average Confidence (Incorrect): {avg_conf_incorrect:.4f}")
    report.append("")
    report.append("Per-Class Error Distribution:")
    report.append(f"{'Class Name':<15} | {'False Positives (FP)':<20} | {'False Negatives (FN)':<20}")
    report.append("-" * 61)
    for cls in classes:
        report.append(f"{cls:<15} | {fp_counts[cls]:<20} | {fn_counts[cls]:<20}")
        
    report.append("\nTop 15 Most Confident Misclassifications (Model is wrong but highly confident):")
    report.append("-" * 90)
    report.append(f"{'Filename':<25} | {'True Class':<12} | {'Predicted':<12} | {'Conf':<6} | Probabilities")
    report.append("-" * 90)
    for f in failures[:15]:
        probs_str = ", ".join([f"{k}: {v:.2f}" for k, v in f['probabilities'].items()])
        report.append(f"{f['filename'][:25]:<25} | {f['true_class']:<12} | {f['pred_class']:<12} | {f['confidence']:.4f} | {probs_str}")
        
    report.append("\n" + "="*70)
    report.append("END OF REPORT")
    
    report_text = "\n".join(report)
    
    # Save report
    report_path = os.path.join(Config.REPORTS_DIR, f"{model_name}_failure_analysis.txt")
    os.makedirs(os.path.dirname(report_path), exist_ok=True)
    with open(report_path, 'w', encoding='utf-8') as file:
        file.write(report_text)
        
    logger.info(f"Saved failure analysis report to {report_path}")
    return failures


Writing /kaggle/working/src/evaluate.py


In [15]:
%%writefile /kaggle/working/src/model.py
import torch
import torch.nn as nn
import timm
from src.config import Config

class CNNBaseline(nn.Module):
    """CNN baseline classifier using a pretrained backbone."""
    def __init__(self, num_classes=Config.NUM_CLASSES):
        super().__init__()
        self.model = timm.create_model(
            Config.CNN_BACKBONE, 
            pretrained=True, 
            num_classes=num_classes
        )

    def forward(self, x):
        return self.model(x)


class ViTBaseline(nn.Module):
    """ViT baseline classifier using a pretrained backbone."""
    def __init__(self, num_classes=Config.NUM_CLASSES):
        super().__init__()
        self.model = timm.create_model(
            Config.ViT_BACKBONE, 
            pretrained=True, 
            num_classes=num_classes
        )

    def forward(self, x):
        return self.model(x)


class BasicHybridModel(nn.Module):
    """Concatenation-based late-fusion hybrid CNN-ViT model (reproduces the user's previous draft)."""
    def __init__(self, num_classes=Config.NUM_CLASSES):
        super().__init__()
        # Extractor 1: CNN (no head)
        self.cnn = timm.create_model(Config.CNN_BACKBONE, pretrained=True, num_classes=0)
        # Extractor 2: ViT (no head)
        self.vit = timm.create_model(Config.ViT_BACKBONE, pretrained=True, num_classes=0)
        
        # Determine feature dimensions
        with torch.no_grad():
            dummy = torch.zeros(1, 3, Config.IMAGE_SIZE[0], Config.IMAGE_SIZE[1])
            cnn_dim = self.cnn(dummy).shape[1]
            vit_dim = self.vit(dummy).shape[1]
            
        fused_dim = cnn_dim + vit_dim
        
        # Classifier Head
        self.fc = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        cnn_feat = self.cnn(x)
        vit_feat = self.vit(x)
        fused = torch.cat((cnn_feat, vit_feat), dim=1)
        return self.fc(fused)


class MultiHeadAttention(nn.Module):
    """Custom Multi-head Self-Attention layer designed to explicitly expose attention weights for explainability."""
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        assert self.head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"
        
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        
        self.attention_weights = None  # To store weights for visualization

    def forward(self, x, store_attention=False):
        B, N, C = x.shape
        
        # Linear projections & reshape -> (B, num_heads, N, head_dim)
        q = self.q_proj(x).reshape(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).reshape(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).reshape(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Attention scores: Q * K_T / sqrt(d_k)
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)  # (B, num_heads, N, N)
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        if store_attention:
            self.attention_weights = attn.detach().cpu()  # Store weights locally
            
        out = torch.matmul(attn, v)  # (B, num_heads, N, head_dim)
        out = out.transpose(1, 2).reshape(B, N, C)  # (B, N, C)
        
        return self.out_proj(out)


class CustomTransformerBlock(nn.Module):
    """Custom Transformer Encoder Block (Pre-LN style) with attention weights access."""
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x, store_attention=False):
        # Attention branch + Residual
        x = x + self.attn(self.norm1(x), store_attention=store_attention)
        # Feedforward branch + Residual
        x = x + self.mlp(self.norm2(x))
        return x


class ImprovedHybridModel(nn.Module):
    """True Hybrid CNN-Vision Transformer.
    
    1. Extracts intermediate multi-scale feature maps from the CNN backbone.
    2. Projects feature maps to a shared embedding dimension.
    3. Flattens feature maps to tokens.
    4. Appends a learnable [CLS] token and adds learnable positional embeddings.
    5. Feeds tokens into a custom Transformer Encoder.
    6. Uses the output [CLS] representation for classification.
    """
    def __init__(self, num_classes=Config.NUM_CLASSES):
        super().__init__()
        # Pretrained CNN Backbone (without final pooling/classification)
        self.cnn = timm.create_model(Config.CNN_BACKBONE, pretrained=True, features_only=True)
        
        # Determine intermediate shape dynamically by passing a dummy tensor
        with torch.no_grad():
            dummy = torch.zeros(1, 3, Config.IMAGE_SIZE[0], Config.IMAGE_SIZE[1])
            dummy_features = self.cnn(dummy)
            
        # Extract features from the last two stages
        self.stage3_idx = len(dummy_features) - 2  # Stage 3
        self.stage4_idx = len(dummy_features) - 1  # Stage 4
        
        c3 = dummy_features[self.stage3_idx].shape[1]
        h3, w3 = dummy_features[self.stage3_idx].shape[2], dummy_features[self.stage3_idx].shape[3]
        
        c4 = dummy_features[self.stage4_idx].shape[1]
        h4, w4 = dummy_features[self.stage4_idx].shape[2], dummy_features[self.stage4_idx].shape[3]
        
        self.num_tokens_stage3 = h3 * w3  # e.g., 14x14 = 196
        self.num_tokens_stage4 = h4 * w4  # e.g., 7x7 = 49
        self.total_tokens = self.num_tokens_stage3 + self.num_tokens_stage4
        
        embed_dim = Config.HYBRID_EMBED_DIM
        
        # 1x1 convolutions to project different channel dims to embed_dim
        self.proj_stage3 = nn.Conv2d(c3, embed_dim, kernel_size=1)
        self.proj_stage4 = nn.Conv2d(c4, embed_dim, kernel_size=1)
        
        # Positional Embeddings & CLS token
        self.pos_embed = nn.Parameter(torch.zeros(1, self.total_tokens, embed_dim))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        # Transformer Blocks
        self.transformer = nn.ModuleList([
            CustomTransformerBlock(
                embed_dim=embed_dim,
                num_heads=Config.HYBRID_NUM_HEADS,
                mlp_dim=Config.HYBRID_MLP_DIM,
                dropout=Config.HYBRID_DROPOUT
            )
            for _ in range(Config.HYBRID_NUM_LAYERS)
        ])
        
        # Classification Head
        self.cls_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim // 2),
            nn.ReLU(),
            nn.Dropout(Config.HYBRID_DROPOUT),
            nn.Linear(embed_dim // 2, num_classes)
        )
        
        # Weight initialization
        nn.init.normal_(self.pos_embed, std=0.02)
        nn.init.normal_(self.cls_token, std=0.02)

    def forward(self, x, store_attention=False):
        # 1. CNN feature extraction
        features = self.cnn(x)
        f3 = features[self.stage3_idx]
        f4 = features[self.stage4_idx]
        
        # 2. Project stage 3 to (B, embed_dim, H3, W3) and flatten to (B, H3*W3, embed_dim)
        tok3 = self.proj_stage3(f3)
        tok3 = tok3.flatten(2).transpose(1, 2)
        
        # 3. Project stage 4 to (B, embed_dim, H4, W4) and flatten to (B, H4*W4, embed_dim)
        tok4 = self.proj_stage4(f4)
        tok4 = tok4.flatten(2).transpose(1, 2)
        
        # 4. Concatenate tokens (multi-scale feature fusion)
        tokens = torch.cat([tok3, tok4], dim=1)  # (B, total_tokens, embed_dim)
        
        # 5. Add positional embeddings
        tokens = tokens + self.pos_embed
        
        # 6. Prepend CLS token
        B = x.shape[0]
        cls_tokens = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls_tokens, tokens], dim=1)  # (B, total_tokens + 1, embed_dim)
        
        # 7. Pass through Custom Transformer Blocks
        for i, block in enumerate(self.transformer):
            # Only store attention weights in the final layer
            store_layer_attn = store_attention and (i == len(self.transformer) - 1)
            tokens = block(tokens, store_attention=store_layer_attn)
            
        # 8. Classification
        cls_out = tokens[:, 0]  # Take CLS token representation
        logits = self.cls_head(cls_out)
        
        return logits


Writing /kaggle/working/src/model.py


In [16]:
%%writefile /kaggle/working/src/preprocessing.py
import cv2
import numpy as np
import torch
import torchvision.transforms as transforms
from src.config import Config

class MedicalPreprocessing:
    @staticmethod
    def apply_clahe(image):
        """Applies Contrast Limited Adaptive Histogram Equalization (CLAHE) to the L-channel of the LAB image.
        This enhances local contrast while preserving overall intensity.
        """
        # Convert BGR to LAB
        lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        
        # Apply CLAHE to the L channel
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        cl = clahe.apply(l)
        
        # Merge back and convert to BGR
        merged = cv2.merge((cl, a, b))
        enhanced = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)
        return enhanced

    @staticmethod
    def denoise(image):
        """Applies gentle bilateral filtering to denoise the image while preserving tumor boundaries.
        Bilateral filtering is superior to Gaussian blur for medical images because it preserves sharp edges.
        """
        return cv2.bilateralFilter(image, d=5, sigmaColor=75, sigmaSpace=75)

    @staticmethod
    def get_transforms():
        """Returns standard PyTorch torchvision transforms for train and validation/test."""
        # Mean and std from ImageNet
        mean = Config.NORM_MEAN
        std = Config.NORM_STD
        
        # Training transforms include safe, realistic augmentations
        train_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(Config.IMAGE_SIZE),
            transforms.RandomRotation(15),  # safe rotation limit
            transforms.RandomHorizontalFlip(),  # horizontal flipping is anatomically valid
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std)
        ])
        
        # Validation and test transforms: NO augmentations, only resizing and normalization
        val_test_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(Config.IMAGE_SIZE),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std)
        ])
        
        return train_transform, val_test_transform

    @classmethod
    def preprocess_image(cls, image_path, is_training=False):
        """Full image loading and preprocessing pipeline for a single image file."""
        img = cv2.imread(image_path)
        if img is None:
            raise ValueError(f"Image not found or corrupt: {image_path}")
        
        # 1. Apply CLAHE contrast enhancement
        img = cls.apply_clahe(img)
        
        # 2. Denoise (edge-preserving)
        img = cls.denoise(img)
        
        # 3. Standardize dimensions and convert to tensor
        train_trans, val_test_trans = cls.get_transforms()
        transform = train_trans if is_training else val_test_trans
        
        # Convert BGR (OpenCV) to RGB (expected by torchvision PIL/ImageNet backbones)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_tensor = transform(img_rgb)
        
        return img_tensor


Writing /kaggle/working/src/preprocessing.py


In [17]:
%%writefile /kaggle/working/src/utils.py
import os
import random
import numpy as np
import torch
import logging

def set_seed(seed=42):
    """Sets random seeds for reproducibility across random, numpy, and PyTorch."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Random seed set to {seed}")

def setup_logger(log_file=None):
    """Sets up a console and file logger for tracking experiments."""
    logger = logging.getLogger()
    logger.setLevel(logging.INFO)
    
    # Avoid duplicate handlers if logger is re-initialized
    if logger.hasHandlers():
        logger.handlers.clear()
        
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    
    # Console Handler
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)
    
    # File Handler (Optional)
    if log_file:
        os.makedirs(os.path.dirname(log_file), exist_ok=True)
        file_handler = logging.FileHandler(log_file)
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)
        
    return logger


Writing /kaggle/working/src/utils.py


In [18]:
!touch /kaggle/working/src/__init__.py

In [19]:
!pip install -q timm tabulate

In [20]:
!ls /kaggle/working/
!ls /kaggle/working/src/

main.py  __notebook__.ipynb  src
config.py   evaluate.py        __init__.py  preprocessing.py  utils.py
dataset.py  explainability.py  model.py     train.py


In [21]:
%cd /kaggle/working
!python main.py --mode dry_run --model improved_hybrid

/kaggle/working
Random seed set to 42

DATASET ANALYSIS & DE-DUPLICATION SUMMARY
Training images found (before de-dup): 5600
Training images unique: 5429
Training duplicates removed: 171
Testing images found (before de-dup): 1600
Testing images unique: 1584
Testing duplicates removed: 16

Class Distribution (De-duplicated):
Class Name      | Training Count  | Testing Count  
-------------------------------------------------------
glioma          | 1400            | 386            
meningioma      | 1386            | 398            
notumor         | 1281            | 400            
pituitary       | 1362            | 400            

Stratified Split:
  Training Set Size:   4343
  Validation Set Size: 1086
  Testing Set Size:    1584
2026-08-22 19:44:54 - INFO - Calculated class weights for loss function: [0.96941966 0.97903514 1.0592682  0.9970156 ]
2026-08-22 19:44:54 - INFO - 
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
DRY RUN MODE ENABLED - Training on tiny subset for 1 epoch
!!!!!!

In [22]:
%cd /kaggle/working
!python main.py --mode train_eval --model improved_hybrid --epochs 25

/kaggle/working
Random seed set to 42

DATASET ANALYSIS & DE-DUPLICATION SUMMARY
Training images found (before de-dup): 5600
Training images unique: 5429
Training duplicates removed: 171
Testing images found (before de-dup): 1600
Testing images unique: 1584
Testing duplicates removed: 16

Class Distribution (De-duplicated):
Class Name      | Training Count  | Testing Count  
-------------------------------------------------------
glioma          | 1400            | 386            
meningioma      | 1386            | 398            
notumor         | 1281            | 400            
pituitary       | 1362            | 400            

Stratified Split:
  Training Set Size:   4343
  Validation Set Size: 1086
  Testing Set Size:    1584
2026-08-22 19:46:34 - INFO - Calculated class weights for loss function: [0.96941966 0.97903514 1.0592682  0.9970156 ]
2026-08-22 19:46:34 - INFO - Loading pretrained weights from Hugging Face hub (timm/resnet18.a1_in1k)
2026-08-22 19:46:34 - INFO - HTTP 